#Construir a Dimensão de Fabricantes (Constructors Dimension)

- 1.Ler a tabela constructors da camada silver
- 2.Ler a tabela ref_nationality_region da camada gold
- 3.Juntar (join) os dados de constructors com ref_nationality_region usando nationality
- 4.Selecionar as colunas necessárias:
> 	•constructors.constructor_id
> 	•constructors.constructor_name
> 	•constructors.nationality
> 	•ref_nationality_region.region
- 5.Escrever os dados transformados na tabela dim_constructors da camada gold

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
from pyspark.sql import functions as F

Passo 1: 
- Ler a tabela constructors da camada silver
- Ler a tabela ref_nationality_region da camada gold

In [0]:
constructors_df = (
    spark.table(f"{catalog_name}.{silver_schema}.constructors")
    .filter((F.col("batch_id")== v_batch_id))
   )

In [0]:
ref_nationality_region = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

Passo2:
- Juntar (join) os dados de constructors com ref_nationality_region usando nationality
- Selecionar as colunas necessárias:

> •constructors.constructor_id
> •constructors.constructor_name
> •constructors.nationality
> •ref_nationality_region.region

In [0]:
dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region,
            constructors_df.nationality == ref_nationality_region.nationality,
            "left"
        )
        .select(
            constructors_df.constructor_id,
            constructors_df.constructor_name,
            constructors_df.nationality,
            ref_nationality_region.region.alias("nationality_region")
        )
)

In [0]:
display(dim_constructors_df)

Escrever os dados transformados na tabela dim_constructors da camada gold

In [0]:
write_to_gold(
    input_df=dim_constructors_df,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)

In [0]:
display(spark.table(target_table))

constructor_id,constructor_name,nationality,nationality_region,created_timestamp,updated_timestamp
ats,ATS,Italian,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
benetton,Benetton,Italian,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
bmw,BMW,German,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
brabham-repco,Brabham-Repco,British,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
cadillac,Cadillac F1 Team,American,North America,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
force_india,Force India,Indian,Asia,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
lotus-pw,Lotus-Pratt & Whitney,British,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
osella,Osella,Italian,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
token,Token,British,Europe,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
amon,Amon,New Zealander,Oceania,2026-09-12T19:28:33.758Z,2026-09-12T19:28:33.758Z
